<a href="https://colab.research.google.com/github/sans0726/100-days-of-deep-learning/blob/main/Welcome_to_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# Input sequence
X = np.array([1.0, 2.0, 3.0])

# Target sequence
Y = np.array([2.0, 4.0, 6.0])

print("X:", X)
print("Y:", Y)

X: [1. 2. 3.]
Y: [2. 4. 6.]


In [2]:
np.random.seed(42)

input_size = 1
hidden_size = 3
output_size = 1

# RNN parameters
W_xh = np.random.randn(hidden_size, input_size) * 0.1
W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
b_h = np.zeros((hidden_size, 1))

# Output layer
W_hy = np.random.randn(output_size, hidden_size) * 0.1
b_y = np.zeros((output_size, 1))

print("W_xh:\n", W_xh)
print("\nW_hh:\n", W_hh)
print("\nW_hy:\n", W_hy)

W_xh:
 [[ 0.04967142]
 [-0.01382643]
 [ 0.06476885]]

W_hh:
 [[ 0.15230299 -0.02341534 -0.0234137 ]
 [ 0.15792128  0.07674347 -0.04694744]
 [ 0.054256   -0.04634177 -0.04657298]]

W_hy:
 [[ 0.02419623 -0.19132802 -0.17249178]]


In [3]:
h_prev = np.zeros((hidden_size, 1))

hidden_states = []
outputs = []

for t in range(len(X)):

    x_t = np.array([[X[t]]])

    # Hidden state
    h_t = np.tanh(
        W_xh @ x_t +
        W_hh @ h_prev +
        b_h
    )

    # Output
    y_t = W_hy @ h_t + b_y

    hidden_states.append(h_t)
    outputs.append(y_t)

    h_prev = h_t

print("Outputs:")

for t, y in enumerate(outputs):
    print(f"t={t}: {y.flatten()}")

Outputs:
t=0: [-0.00731041]
t=1: [-0.01515192]
t=2: [-0.0230725]


In [4]:
loss = 0

for t in range(len(X)):
    target = Y[t]
    prediction = outputs[t][0, 0]

    loss += 0.5 * (prediction - target) ** 2

print("Loss:", loss)

Loss: 28.214071142630537


In [5]:
# Gradient for W_hy
dW_hy = np.zeros_like(W_hy)

for t in range(len(X)):
    y_t = outputs[t]
    h_t = hidden_states[t]

    # dL/dy
    dy = y_t - Y[t]

    # dL/dW_hy
    dW_hy += dy * h_t.T

print("Gradient dW_hy:")
print(dW_hy)

Gradient dW_hy:
[[-1.49325409  0.32089855 -1.80882137]]


In [6]:
# Gradient of loss with respect to hidden states

dh_next = np.zeros((hidden_size, 1))

for t in reversed(range(len(X))):

    y_t = outputs[t]
    h_t = hidden_states[t]

    # Gradient from output
    dy = y_t - Y[t]

    # Gradient flowing from output to hidden state
    dh = W_hy.T @ dy

    # Add gradient coming from the next timestep
    dh += dh_next

    # Gradient through tanh
    da = dh * (1 - h_t ** 2)

    print(f"\nt = {t}")
    print("dh:")
    print(dh)
    print("da:")
    print(da)

    # Gradient passed to previous hidden state
    dh_next = W_hh.T @ da


t = 2
dh:
[[-0.14573563]
 [ 1.15238256]
 [ 1.03893052]]
da:
[[-0.14194988]
 [ 1.15114786]
 [ 1.00036134]]

t = 1
dh:
[[0.11729544]
 [0.81351946]
 [0.59527104]]
da:
[[0.11599438]
 [0.81305445]
 [0.58534457]]

t = 0
dh:
[[0.12925401]
 [0.41660941]
 [0.27809663]]
da:
[[0.12893563]
 [0.41652977]
 [0.27693327]]


In [7]:
# Initialize gradients
dW_xh = np.zeros_like(W_xh)
dW_hh = np.zeros_like(W_hh)
db_h = np.zeros_like(b_h)

# Gradient flowing from the future hidden state
dh_next = np.zeros((hidden_size, 1))

# Backpropagate through time
for t in reversed(range(len(X))):

    x_t = np.array([[X[t]]])
    h_t = hidden_states[t]

    # Previous hidden state
    if t == 0:
        h_prev = np.zeros((hidden_size, 1))
    else:
        h_prev = hidden_states[t - 1]

    # Output gradient
    dy = outputs[t] - Y[t]

    # Gradient from output layer to hidden state
    dh = W_hy.T @ dy

    # Add gradient coming from future timestep
    dh += dh_next

    # Backpropagate through tanh
    da = dh * (1 - h_t ** 2)

    # Gradients for RNN parameters
    dW_xh += da @ x_t.T
    dW_hh += da @ h_prev.T
    db_h += da

    # Pass gradient to previous hidden state
    dh_next = W_hh.T @ da


print("dW_xh:")
print(dW_xh)

print("\ndW_hh:")
print(dW_hh)

print("\ndb_h:")
print(db_h)

dW_xh:
[[-0.06492526]
 [ 5.49608226]
 [ 4.44870642]]

dW_hh:
[[-0.00919316  0.00179006 -0.0108282 ]
 [ 0.16159021 -0.03876266  0.20123923]
 [ 0.13440813 -0.03200943  0.16703967]]

db_h:
[[0.10298013]
 [2.38073209]
 [1.86263918]]
